# EX3 — LoCoMotif vs. Matrix-Profile family on Leno UI logs

Comparison of five motif-discovery approaches on the four Leno SmartRPA logs:

- **LoCoMotif** — multivariate, z-normed Word2Vec.
- **MStampBrute** *(round 2)* — `stumpy.mstump` swept over every `m ∈ [l_min, l_max]` on the **same multivariate input as LoCoMotif**. Added to control for the PCA(1) dimensionality starvation that handicaps the univariate baselines.
- **Brute-force MP** — univariate `stumpy.stump` swept over every `m`, PCA(1) of the Word2Vec matrix. Retained as the PCA(1) ablation.
- **Pan Matrix Profile** — `stumpy.stimp` over `m ∈ [l_min, l_max]`, both `percentage=1.0` (full SKIMP) and `percentage=0.01` (sampled SKIMP). Round 2 extracts motifs from **every row of the pan structure** and pools across `m` (the round-1 implementation collapsed to a single best-m and re-extracted via `stumpy.stump`, which was the bug that produced identical `PMP_full`/`PMP_sampled` F1 — fixed).
- **Random** *(round 2)* — chance baseline: `k` segments per seed with lengths Uniform[l_min, l_max] and uniform starts, averaged over 100 seeds. Reports F1 mean ± std so the chance-corrected gain `F1 − F1_random` is visible.

All methodological choices and their paper-facing justification are documented in `EX3_design_decisions.md` next to this notebook. The runner module `ex3_runner.py` exposes the algorithm wrappers; this notebook is only the driver.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

from JupyterNotebooks import ex3_runner
print('Logs to process:', ex3_runner.LENO_LOGS)

/Users/tom/vsCode/TSMDforUILogs/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Logs to process: ['202511_SR_RT_plus', '202511_SR_RT_plus_extended', '202511_SR_RT_parallel', '202511_SR_RT_parallel_extended']


## Run all four logs

For each log, `run_all_for_log` performs:
1. Load CSV + ground-truth (Leno schema: targetApp, url, target.workbookName, target.sheetName, target.id, target.tagName, target.type, target.name, target.href, eventType).
2. Word2Vec encode at `vector_size = round(sqrt(unique_tokens))` (same encoder as `experiment.py`).
3. Per-dim z-norm → PCA(1) for the univariate-MP inputs; explained-variance ratio reported. The multivariate baselines (LoCoMotif, MStampBrute) consume the z-normed matrix directly.
4. Run **LoCoMotif** and **MStampBrute** on multivariate z-normed W2V, **BruteMP** on PCA(1), **PMP_full** / **PMP_sampled** on PCA(1) (now extracting from the full pan structure), and **Random** (100-seed chance baseline).
5. Score with IoU ≥ 0.8 + one-to-one matching (`evaluate_motifs(overlap_type='iou')`).

Oracle bounds: `l_min = min(GT lengths)`, `l_max = max(GT lengths)`, `k = |GT|` — same for every algorithm including Random. See `EX3_design_decisions.md` §5 and §11.

In [2]:
results = ex3_runner.run_all_logs()
results.to_csv('ex3_results.csv', index=False)
print(f'\nSaved {len(results)} rows to ex3_results.csv')
results


=== 202511_SR_RT_plus ===
[202511_SR_RT_plus] T=4646 d=27 l_min=28 l_max=65 k=100 pca_ev=0.557

=== 202511_SR_RT_plus_extended ===
[202511_SR_RT_plus_extended] T=9646 d=29 l_min=28 l_max=65 k=100 pca_ev=0.281

=== 202511_SR_RT_parallel ===
[202511_SR_RT_parallel] T=4646 d=27 l_min=28 l_max=63 k=100 pca_ev=0.436

=== 202511_SR_RT_parallel_extended ===
[202511_SR_RT_parallel_extended] T=9596 d=29 l_min=29 l_max=63 k=100 pca_ev=0.279

Saved 24 rows to ex3_results.csv


,log_name,log_length,n_gt_motifs,algorithm,l_min,l_max,w2v_dim,pca_explained_variance,preprocessing_time_s,discovery_time_s,best_m,tp,fp,fn,precision,recall,f1,mean_iou_tp,n_discovered,f1_std,precision_std,recall_std,n_seeds
0,202511_SR_RT_plus,4646,100,LoCoMotif,28,65,27,0.557033,0.122304,5.598004,NaN,46.00,54.00,54.00,0.460000,0.4600,0.460000,0.968195,100,NaN,NaN,NaN,NaN
1,202511_SR_RT_plus,4646,100,BruteMP,28,65,27,0.557033,0.122304,7.618031,NaN,8.00,92.00,92.00,0.080000,0.0800,0.080000,0.880897,100,NaN,NaN,NaN,NaN
2,202511_SR_RT_plus,4646,100,MStampBrute,28,65,27,0.557033,0.122304,828.192507,NaN,6.00,27.00,94.00,0.181818,0.0600,0.090226,0.864266,33,NaN,NaN,NaN,NaN
3,202511_SR_RT_plus,4646,100,PMP_full,28,65,27,0.557033,0.122304,2.615134,47.0,3.00,97.00,97.00,0.030000,0.0300,0.030000,0.861589,100,NaN,NaN,NaN,NaN
4,202511_SR_RT_plus,4646,100,PMP_sampled,28,65,27,0.557033,0.122304,6.970314,47.0,4.00,96.00,96.00,0.040000,0.0400,0.040000,0.876030,100,NaN,NaN,NaN,NaN
5,202511_SR_RT_plus,4646,100,Random,28,65,27,0.557033,0.122304,2.924017,NaN,7.89,92.11,92.11,0.078900,0.0789,0.078900,0.872392,100,0.024200,0.024200,0.024200,100.0
6,202511_SR_RT_plus_extended,9646,100,LoCoMotif,28,65,29,0.281394,0.307129,49.241233,NaN,50.00,50.00,50.00,0.500000,0.5000,0.500000,0.928901,100,NaN,NaN,NaN,NaN
7,202511_SR_RT_plus_extended,9646,100,BruteMP,28,65,29,0.281394,0.307129,3.733912,NaN,10.00,90.00,90.00,0.100000,0.1000,0.100000,0.900803,100,NaN,NaN,NaN,NaN
8,202511_SR_RT_plus_extended,9646,100,MStampBrute,28,65,29,0.281394,0.307129,2285.625237,NaN,8.00,27.00,92.00,0.228571,0.0800,0.118519,0.879160,35,NaN,NaN,NaN,NaN
9,202511_SR_RT_plus_extended,9646,100,PMP_full,28,65,29,0.281394,0.307129,4.637786,47.0,2.00,98.00,98.00,0.020000,0.0200,0.020000,0.863363,100,NaN,NaN,NaN,NaN


## Table 1 — Discovery time (s)

In [3]:
time_table = results.pivot_table(
    index='log_name', columns='algorithm', values='discovery_time_s'
).round(2)
time_table

algorithm,BruteMP,LoCoMotif,MStampBrute,PMP_full,PMP_sampled,Random
log_name,,,,,,
202511_SR_RT_parallel,2.19,5.97,772.85,2.49,2.99,3.03
202511_SR_RT_parallel_extended,3.55,42.64,2107.74,4.17,5.01,2.92
202511_SR_RT_plus,7.62,5.60,828.19,2.62,6.97,2.92
202511_SR_RT_plus_extended,3.73,49.24,2285.63,4.64,5.56,3.04


## Table 2 — Precision / Recall / F1 (IoU ≥ 0.8, one-to-one)

In [4]:
metric_table = results.pivot_table(
    index='log_name', columns='algorithm',
    values=['precision', 'recall', 'f1'],
).round(3)
metric_table

f1                                                   precision                                                    recall                                 \
algorithm                      BruteMP LoCoMotif MStampBrute PMP_full PMP_sampled Random   BruteMP LoCoMotif MStampBrute PMP_full PMP_sampled Random BruteMP LoCoMotif MStampBrute PMP_full   
log_name                                                                                                                                                                                      
202511_SR_RT_parallel             0.09     0.995       0.016     0.08        0.09  0.090      0.09      1.00       0.034     0.08        0.09  0.090    0.09      0.99        0.01     0.08   
202511_SR_RT_parallel_extended    0.07     0.500       0.104     0.02        0.01  0.041      0.07      0.50       0.200     0.02        0.01  0.041    0.07      0.50        0.07     0.02   
202511_SR_RT_plus                 0.08     0.460       0.090     0.03        0.04  0.079      0.08      0.46       0.182     0.03        0.04  0.079    0.08      0.46        0.06     0.03   
202511_SR_RT_plus_extended        0.10     0.500       0.119     0.02        0.03  0.037      0.10      0.50       0.229     0.02        0.03  0.037    0.10      0.50        0.08     0.02   

                                                   
algorithm                      PMP_sampled Random  
log_name                                           
202511_SR_RT_parallel                 0.09  0.090  
202511_SR_RT_parallel_extended        0.01  0.041  
202511_SR_RT_plus                     0.04  0.079  
202511_SR_RT_plus_extended            0.03  0.037

## PCA(1) explained variance per log

Reported for transparency — it bounds how much information the MP-family inputs lose vs. LoCoMotif's multivariate input. See `EX3_design_decisions.md` §4.

In [5]:
pca_table = results[['log_name', 'log_length', 'n_gt_motifs',
                     'l_min', 'l_max', 'w2v_dim',
                     'pca_explained_variance', 'preprocessing_time_s']
                    ].drop_duplicates(subset=['log_name']).set_index('log_name').round(3)
pca_table

,log_length,n_gt_motifs,l_min,l_max,w2v_dim,pca_explained_variance,preprocessing_time_s
log_name,,,,,,,
202511_SR_RT_plus,4646,100,28,65,27,0.557,0.122
202511_SR_RT_plus_extended,9646,100,28,65,29,0.281,0.307
202511_SR_RT_parallel,4646,100,28,63,27,0.436,0.127
202511_SR_RT_parallel_extended,9596,100,29,63,29,0.279,0.309
